## Use the generate_chat.py and try to pivot the chat messages

In [1]:
import pandas as pd
from generator.generate_chat import *

df = generate_chat_data_v1(num_to_generate=100)
df.head()

,room,username,message,timestamp
0,Room4,UserC,1. Delta1,2026-04-01 14:34:12.443340
1,Room4,UserC,2. ESCORT9,2026-04-01 14:34:12.456340
2,Room4,UserC,3. TEST,2026-04-01 14:34:12.456340
3,Room4,UserC,4. None,2026-04-01 14:34:12.467340
4,Room4,UserC,5. Expected Completion Time9: 2026-04-01T14:39...,2026-04-01 14:34:12.475340


### Simple naive merge

In [2]:
## That worked, so go ahead and write a function to do this
def pivot_multiline_messages(dataframe, prefixes, new_columns=None, order_by=["Room Name", "Timestamp"], message_column="Message"):
    if dataframe is None:
        raise ValueError("The input DataFrame is empty.")
    if prefixes is None or len(prefixes) == 0:
        raise ValueError("No prefixes provided.")

    ## If new columns weren't passed to us, go ahead and create some new prefixes
    if new_columns is None:
        new_columns = [f"msg_{i+1}" for i in range(len(prefixes))]

    to_return = {}
    tmp = dataframe[dataframe[message_column].str.startswith(prefixes[0])].copy().sort_values(by=order_by)
    for col in order_by:
        to_return[col] = tmp[col].values

    for i, prefix in enumerate(prefixes):
        tmp = dataframe[dataframe[message_column].str.startswith(prefix)].copy().sort_values(by=order_by)
        to_return[new_columns[i]] = tmp[message_column].str.slice(len(prefix)).values

    ## Return the our results
    return pd.DataFrame(to_return)

In [3]:
prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]

pivot_multiline_messages(df, prefixes, new_columns=new_columns, order_by=["room", "timestamp"], message_column="message")

,room,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5
0,Room1,2026-04-01 14:41:38.510340,Echo4,N/A10,SCIKIT,None,Time to Completion1: 2026-04-01T14:42:31.542340
1,Room1,2026-04-01 15:02:15.745340,Delta4,ESCORT5,PANDAS,None,TTC2: 2026-04-01T15:11:29.749340
2,Room1,2026-04-01 15:18:13.801340,Echo5,ESCORT7,SCIKIT,N/A,TTC10: 2026-04-01T15:22:59.812340
3,Room1,2026-04-01 15:50:17.044340,Tango1,ESCORT1,NUMPY,N/A,TTC5: 2026-04-01T15:51:49.081340
4,Room1,2026-04-01 16:06:24.197340,Bravo2,ESCORT2,TEST,N/A,TTC5: 2026-04-01T16:13:22.236340
...,...,...,...,...,...,...,...
95,Room5,2026-04-01 22:28:04.727340,Echo4,TEST4,PANDAS,None,TTC5: 2026-04-01T22:31:36.768340
96,Room5,2026-04-01 23:06:27.062340,Bravo3,N/A4,TEST,N/A,ETA6: 2026-04-01T23:10:13.080340
97,Room5,2026-04-01 23:10:13.093340,Tango10,SMACK10,SCIKIT,N/A,Expected Completion Time10: 2026-04-01T23:11:3...
98,Room5,2026-04-01 23:11:34.141340,Tango10,N/A7,SCIKIT,None,TTC8: 2026-04-01T23:15:48.171340


### Simple naive merge v2

In [4]:
## That worked, so go ahead and write a function to do this
def pivot_multiline_messages(dataframe, prefixes, new_columns=None, order_by=["Room Name"], time_column="Timestamp", message_column="Message"):
    if dataframe is None:
        raise ValueError("The input DataFrame is empty.")
    if prefixes is None or len(prefixes) == 0:
        raise ValueError("No prefixes provided.")

    ## Add the timestamp column to the order by
    if order_by is None:
        order_by = [time_column]
    elif time_column in order_by:
        pass
    else:
        order_by.append(time_column)

    ## If new columns weren't passed to us, go ahead and create some new prefixes
    if new_columns is None:
        new_columns = [f"msg_{i+1}" for i in range(len(prefixes))]

    ## Create our to return
    to_return = {}

    ## Add the order by column(s) to our results,
    tmp = dataframe[dataframe[message_column].str.startswith(prefixes[0])].copy().sort_values(by=order_by)
    for col in order_by:
        to_return[col] = tmp[col].values
    ## Add the prefixes that we are looking for the the results
    for i, prefix in enumerate(prefixes):
        tmp = dataframe[dataframe[message_column].str.startswith(prefix)].copy().sort_values(by=order_by)
        to_return[new_columns[i]] = tmp[message_column].str.slice(len(prefix)).values

    ## Return the our results
    return pd.DataFrame(to_return)

In [5]:
prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]

pivot_multiline_messages(df, prefixes, new_columns=new_columns, order_by=["room", "timestamp"], time_column="timestamp", message_column="message")

,room,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5
0,Room1,2026-04-01 14:41:38.510340,Echo4,N/A10,SCIKIT,None,Time to Completion1: 2026-04-01T14:42:31.542340
1,Room1,2026-04-01 15:02:15.745340,Delta4,ESCORT5,PANDAS,None,TTC2: 2026-04-01T15:11:29.749340
2,Room1,2026-04-01 15:18:13.801340,Echo5,ESCORT7,SCIKIT,N/A,TTC10: 2026-04-01T15:22:59.812340
3,Room1,2026-04-01 15:50:17.044340,Tango1,ESCORT1,NUMPY,N/A,TTC5: 2026-04-01T15:51:49.081340
4,Room1,2026-04-01 16:06:24.197340,Bravo2,ESCORT2,TEST,N/A,TTC5: 2026-04-01T16:13:22.236340
...,...,...,...,...,...,...,...
95,Room5,2026-04-01 22:28:04.727340,Echo4,TEST4,PANDAS,None,TTC5: 2026-04-01T22:31:36.768340
96,Room5,2026-04-01 23:06:27.062340,Bravo3,N/A4,TEST,N/A,ETA6: 2026-04-01T23:10:13.080340
97,Room5,2026-04-01 23:10:13.093340,Tango10,SMACK10,SCIKIT,N/A,Expected Completion Time10: 2026-04-01T23:11:3...
98,Room5,2026-04-01 23:11:34.141340,Tango10,N/A7,SCIKIT,None,TTC8: 2026-04-01T23:15:48.171340


### Try to perform a better merge, that isn't so naive about the timestamp and order

In [6]:

prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]

new_df = None
for i, prefix in enumerate(prefixes):
    tmpdf = df[df["message"].str.startswith(prefix)].copy()
    tmpdf[new_columns[i]] = tmpdf["message"].str.slice(len(prefix)).values
    tmpdf = tmpdf.drop(columns=["message"])

    if new_df is None:
        new_df = tmpdf
    else:
        new_df = pd.merge(new_df, tmpdf, on=["room", "timestamp", "username"], how="outer")

display(new_df)


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5
0,Room1,UserB,2026-04-01 14:41:38.510340,Echo4,NaN,NaN,NaN,NaN
1,Room1,UserB,2026-04-01 14:41:38.519340,NaN,N/A10,NaN,NaN,NaN
2,Room1,UserB,2026-04-01 14:41:38.529340,NaN,NaN,SCIKIT,NaN,NaN
3,Room1,UserB,2026-04-01 14:41:38.531340,NaN,NaN,NaN,None,NaN
4,Room1,UserB,2026-04-01 14:41:38.542340,NaN,NaN,NaN,NaN,Time to Completion1: 2026-04-01T14:42:31.542340
...,...,...,...,...,...,...,...,...
459,Room5,UserA,2026-04-01 23:11:34.171340,NaN,NaN,NaN,NaN,TTC8: 2026-04-01T23:15:48.171340
460,Room5,UserC,2026-04-01 23:18:38.199340,Bravo5,NaN,NaN,NaN,NaN
461,Room5,UserC,2026-04-01 23:18:38.202340,NaN,ESCORT6,TEST,NaN,NaN
462,Room5,UserC,2026-04-01 23:18:38.211340,NaN,NaN,NaN,N/A,NaN


### Write a function to try and fix the match, to be a little better

In [7]:
def convert_messages_to_ten_line(msg_df,
            prefixes=["1. ", "2. ", "3. ", "4. ", "5. ",
                      "6. ", "7. ", "8. ", "9. ", "10. "],
            new_columns=["msg_1", "msg_2", "msg_3", "msg_4", "msg_5",
                         "msg_6", "msg_7", "msg_8", "msg_9", "msg_10"],
            best_matches=False,
            msg_col="Message", timestmp_col="Timestamp",
            groupby_cnt_col="msg_cnt",
            match_group_col="match_group", time_diff_col="time_diff",
            groupby=["Room Name", "User Name"]):

    ## First go ahead and create the new columns and strip off our prefixes
    new_df = msg_df.copy()
    ## Loop through the prefixes and create the new columns
    for i, prefix in enumerate(prefixes):
        new_df[new_columns[i]] = new_df[new_df[msg_col].str.startswith(prefix)][msg_col].str.slice(len(prefix))

    ## Drop the message column, since we don't need it anymore
    new_df = new_df.drop(columns=[msg_col])

    ## Build our groupby (that includes the timestamp column)
    full_groupby = groupby.copy()
    full_groupby.append(timestmp_col)

    ## Add a column for our groupby count
    new_df[groupby_cnt_col] = 0

    ## GroupBy, first build a dic for our group by
    agg_dict = {}
    for col in new_df.columns:
        if not (col in full_groupby):
          agg_dict[col] = "first"
    agg_dict[groupby_cnt_col] = "count"
    new_df = new_df.groupby(full_groupby).agg(agg_dict).reset_index()


    ## Add the new columns we need for building / calculating matches for our dataframes
    new_df[time_diff_col] = [{} for _ in range(len(new_df))]
    new_df[match_group_col] = 0

    ## If we have rows that equal the number of message we were looking for, then set them aside
    complete_df = new_df[new_df[groupby_cnt_col] == len(prefixes)]

    new_df = new_df[new_df[groupby_cnt_col] < len(prefixes)]
    print("\n")
    print(f"==================== partial df {len(new_df)} records ====================")
    display(new_df)

    ## ======== Calculate the time difference between matching rows ========
    ## Loop through all of the rows and calculate the different time differences between the
    ##     rows that were grouped together
    for index, row in new_df.iterrows():
        #print(f"index: {index}")
        time_differnces = {}
        for index2, row2 in new_df[index + 1:].iterrows():
            #print(f"index2: {index2}")
            if index != index2:
                correct_group = True
                for col in groupby:
                    #print(f"{index}.{row[col]} != {index2}.{row2[col]} ({row[col] != row2[col]})")
                    if row2[col] != row[col]:
                        correct_group = False
                        break

                if correct_group:
                    time_differnces[index2] = (row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000
                    new_df.at[index, match_group_col] += 1
                    new_df.at[index, time_diff_col][index2] = abs((row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000)
                    new_df.at[index2, match_group_col] += 1
                    new_df.at[index2, time_diff_col][index] = abs((row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000)

    ## Take any records that had no matches and move them to the complete dataframe
    tmp_df = new_df[new_df[match_group_col] == 0]
    complete_df = pd.concat([complete_df, tmp_df])

    ## Drop the records where there was no match found
    new_df = new_df[new_df[match_group_col] > 0]

    print("\n")
    print(f"==================== unmatched/complete df {len(complete_df)} records, with timedifferences ====================")
    display(complete_df)

    print("\n")
    print(f"==================== partial df {len(new_df)} records, with timedifferences ====================")
    display(new_df)


    ## ======== Loop through all of our unmatched records and see about ========
    ## Setup our merged dataframe
    merged = {}
    for column in new_df.columns:
        merged[column] = []
    rows_inspected = []
    for index, row in new_df.iterrows():
        ## If we've already used this row, then move along
        if index not in rows_inspected:
            ## Matching Row
            time_differnces = row[time_diff_col]
            min_time_diff = min(time_differnces.values())
            key = [key for key, value in time_differnces.items() if value == min_time_diff][0]
            match_row = new_df.loc[key]

            #rows_inspected.append(index)

    return None

prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]
convert_messages_to_ten_line(
    df, prefixes=prefixes, new_columns=new_columns, 
    groupby=["room", "username"], timestmp_col="timestamp", msg_col="message"
)



==================== partial df 464 records ====================


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5,msg_cnt,time_diff,match_group
0,Room1,UserA,2026-04-01 15:50:17.044340,Tango1,NaN,NaN,NaN,NaN,1,{},0
1,Room1,UserA,2026-04-01 15:50:17.053340,NaN,ESCORT1,NaN,NaN,NaN,1,{},0
2,Room1,UserA,2026-04-01 15:50:17.066340,NaN,NaN,NUMPY,N/A,NaN,2,{},0
3,Room1,UserA,2026-04-01 15:50:17.081340,NaN,NaN,NaN,NaN,TTC5: 2026-04-01T15:51:49.081340,1,{},0
4,Room1,UserA,2026-04-01 16:52:04.430340,Charlie5,NaN,NaN,NaN,NaN,1,{},0
...,...,...,...,...,...,...,...,...,...,...,...
459,Room5,UserE,2026-04-01 23:10:13.093340,Tango10,NaN,NaN,NaN,NaN,1,{},0
460,Room5,UserE,2026-04-01 23:10:13.101340,NaN,SMACK10,NaN,NaN,NaN,1,{},0
461,Room5,UserE,2026-04-01 23:10:13.112340,NaN,NaN,SCIKIT,NaN,NaN,1,{},0
462,Room5,UserE,2026-04-01 23:10:13.117340,NaN,NaN,NaN,N/A,NaN,1,{},0




==================== unmatched/complete df 0 records, with timedifferences ====================


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5,msg_cnt,time_diff,match_group




==================== partial df 464 records, with timedifferences ====================


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5,msg_cnt,time_diff,match_group
0,Room1,UserA,2026-04-01 15:50:17.044340,Tango1,NaN,NaN,NaN,NaN,1,"{1: 9.0, 2: 22.0, 3: 37.0, 4: 3707386.0, 5: 37...",23
1,Room1,UserA,2026-04-01 15:50:17.053340,NaN,ESCORT1,NaN,NaN,NaN,1,"{0: 9.0, 2: 13.0, 3: 28.0, 4: 3707377.0, 5: 37...",23
2,Room1,UserA,2026-04-01 15:50:17.066340,NaN,NaN,NUMPY,N/A,NaN,2,"{0: 22.0, 1: 13.0, 3: 15.0, 4: 3707364.0, 5: 3...",23
3,Room1,UserA,2026-04-01 15:50:17.081340,NaN,NaN,NaN,NaN,TTC5: 2026-04-01T15:51:49.081340,1,"{0: 37.0, 1: 28.0, 2: 15.0, 4: 3707349.0, 5: 3...",23
4,Room1,UserA,2026-04-01 16:52:04.430340,Charlie5,NaN,NaN,NaN,NaN,1,"{0: 3707386.0, 1: 3707377.0, 2: 3707364.0, 3: ...",23
...,...,...,...,...,...,...,...,...,...,...,...
459,Room5,UserE,2026-04-01 23:10:13.093340,Tango10,NaN,NaN,NaN,NaN,1,"{445: 12970677.0, 446: 12970673.0, 447: 129706...",18
460,Room5,UserE,2026-04-01 23:10:13.101340,NaN,SMACK10,NaN,NaN,NaN,1,"{445: 12970685.0, 446: 12970681.0, 447: 129706...",18
461,Room5,UserE,2026-04-01 23:10:13.112340,NaN,NaN,SCIKIT,NaN,NaN,1,"{445: 12970696.0, 446: 12970692.0, 447: 129706...",18
462,Room5,UserE,2026-04-01 23:10:13.117340,NaN,NaN,NaN,N/A,NaN,1,"{445: 12970701.0, 446: 12970697.0, 447: 129706...",18


In [8]:
def convert_messages_to_ten_line(msg_df,
            prefixes=["1. ", "2. ", "3. ", "4. ", "5. ",
                      "6. ", "7. ", "8. ", "9. ", "10. "],
            new_columns=["msg_1", "msg_2", "msg_3", "msg_4", "msg_5",
                         "msg_6", "msg_7", "msg_8", "msg_9", "msg_10"],
            best_matches=False,
            msg_col="Message", timestmp_col="Timestamp",
            groupby_cnt_col="msg_cnt",
            match_group_col="match_group", time_diff_col="time_diff",
            groupby=["Room Name", "Username"]):
    """
    Convert our messages, into a ten line format.  Prefixes and newcolumns are defaulted,
        but you can adjust them to whatever format you want.
        Best matches will only return those matches that are smallest
    """

    ## First go ahead and create the new columns and strip off our prefixes
    new_df = msg_df.copy()
    ## Loop through the prefixes and create the new columns
    for i, prefix in enumerate(prefixes):
        new_df[new_columns[i]] = new_df[new_df[msg_col].str.startswith(prefix)][msg_col].str.slice(len(prefix))

    ## Drop the message column, since we don't need it anymore
    new_df = new_df.drop(columns=[msg_col])
    print(f"==================== new df {len(new_df)} records ====================")
    #display(new_df)

    ## Build our groupby (that includes the timestamp column)
    full_groupby = groupby.copy()
    full_groupby.append(timestmp_col)

    ## Add a column for our groupby count
    new_df[groupby_cnt_col] = 0

    ## GroupBy, first build a dic for our group by
    agg_dict = {}
    for col in new_df.columns:
        if not (col in full_groupby):
          agg_dict[col] = "first"
    agg_dict[groupby_cnt_col] = "count"
    new_df = new_df.groupby(full_groupby).agg(agg_dict).reset_index()
    #print("\n")
    #print(f"==================== groupedby df {len(new_df)} records ====================")
    #display(new_df)

    ## Add the new columns we need for building / calculating matches for our dataframes
    new_df[time_diff_col] = [{} for _ in range(len(new_df))]
    new_df[match_group_col] = 0

    ## If we have rows that equal the number of message we were looking for, then set them aside
    complete_df = new_df[new_df[groupby_cnt_col] == len(prefixes)]
    print(f"==================== complete df {len(complete_df)} records ====================")
    #display(complete_df)

    new_df = new_df[new_df[groupby_cnt_col] < len(prefixes)]
    #print("\n")
    print(f"==================== partial df {len(new_df)} records ====================")
    #display(new_df)

    ## ======== Calculate the time difference between matching rows ========
    ## Loop through all of the rows and calculate the different time differences between the
    ##     rows that were grouped together
    for index, row in new_df.iterrows():
        for index2, row2 in new_df[index + 1:].iterrows():
            #print(f"index2: {index2}")
            if index != index2:
                correct_group = True
                for col in groupby:
                    #print(f"{index}.{row[col]} != {index2}.{row2[col]} ({row[col] != row2[col]})")
                    if row2[col] != row[col]:
                        correct_group = False
                        break

                if correct_group:
                    new_df.at[index, match_group_col] += 1
                    new_df.at[index, time_diff_col][index2] = abs((row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000)
                    new_df.at[index2, match_group_col] += 1
                    new_df.at[index2, time_diff_col][index] = abs((row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000)

    ## Take any records that had no matches and move them to the complete dataframe
    tmp_df = new_df[new_df[match_group_col] == 0]
    print(f"==================== partial, with no matches df {len(tmp_df)} records ====================")
    complete_df = pd.concat([complete_df, tmp_df])

    ## Drop the records where there was no match found
    new_df = new_df[new_df[match_group_col] > 0]
    print(f"==================== partial, with matches df {len(new_df)} records ====================")

    #print("\n")
    #print(f"==================== unmatched/complete df {len(complete_df)} records, with timedifferences ====================")
    #display(complete_df)

    #print("\n")
    #print(f"==================== partial df {len(new_df)} records, with timedifferences ====================")
    #display(new_df)


    ## ======== Loop through all of our unmatched records and see about ========
    ## Setup our merged dataframe
    merged = {}
    for column in new_df.columns:
        merged[column] = []
    rows_inspected = []
    for index, row in new_df.iterrows():
        ## If we've already used this row, then move along
        if index not in rows_inspected:
            ## Matching Row
            time_differnces = row[time_diff_col]
            min_time_diff = min(time_differnces.values())
            key = [key for key, value in time_differnces.items() if value == min_time_diff][0]
            match_row = new_df.loc[key]

            #rows_inspected.append(index)

    return None


# convert_messages_to_ten_line(df, prefixes=prefixes, new_columns=new_columns)
convert_messages_to_ten_line(
    df, prefixes=prefixes, new_columns=new_columns, 
    groupby=["room", "username"], timestmp_col="timestamp", msg_col="message"
)

==================== new df 500 records ====================
==================== complete df 0 records ====================
==================== partial df 464 records ====================
==================== partial, with no matches df 0 records ====================
==================== partial, with matches df 464 records ====================


In [9]:
from generator.tenlinesamplegenerator import *

tmp = TenLineSampleGenerator(appendId=True)
fake_ten_line = tmp.generate_sample_data(100)
fake_ten_line.head()

,Room Name,Username,Message,Timestamp
0,c2_experiment,wasp_test,1. Maple21_1,2026-04-01 00:00:00.000000Z
1,c2_experiment,wasp_test,2. DANCE_1,2026-04-01 00:00:00.000000Z
2,c2_experiment,wasp_test,3. TEST_1,2026-04-01 00:00:00.000000Z
3,c2_experiment,wasp_test,4. APPLE_1,2026-04-01 00:00:00.000000Z
4,c2_experiment,wasp_test,5. N/A_1,2026-04-01 00:00:00.000000Z


In [10]:
from parser.tenlineparser import *

parser = TenLineParser()
results = parser.parse(fake_ten_line)
results[results["msg_cnt"] < 10]

self.time_diff_col: time_diff, match_index: 3, index: 2
self.time_diff_col: time_diff, match_index: 2, index: 3
self.time_diff_col: time_diff, match_index: 8, index: 7
self.time_diff_col: time_diff, match_index: 9, index: 7


KeyError: 9

In [ ]:
results.iloc[102]['time_diff']